# Reference Database Setup — VEP 115 + ANNOVAR
### Genesis Workbench — Disease Biology Module (v2)

One-time setup notebook. Downloads and installs all reference databases needed by the `01_vep_annovar_annotation` pipeline.

> **Prerequisites**
> - **Must run on a classic compute cluster** (not serverless) — requires shell access for `git`, `wget`, `perl`, `samtools`
> - Internet access required for Ensembl FTP, GitHub, and ANNOVAR database downloads
> - **ANNOVAR requires registration** at https://annovar.openbioinformatics.org/ — you must provide your personalized download URL
> - Estimated disk usage: \~50–60 GB on UC Volume (reference FASTA \~3 GB, VEP cache \~17 GB, ANNOVAR databases \~30 GB)
> - Estimated runtime: 45–90 minutes depending on network speed

```
variant_annotation_reference/
├── reference/
│   ├── GRCh38.fa              (~3 GB)
│   └── GRCh38.fa.fai
├── vep_cache/
│   ├── ensembl-vep-release-115/   (VEP Perl source + binary)
│   └── homo_sapiens/
│       └── 115_GRCh38/           (~17 GB cache)
└── annovar/
    ├── annotate_variation.pl
    ├── table_annovar.pl
    ├── convert2annovar.pl
    └── humandb/                   (~30 GB databases)
        ├── hg38_refGene.txt
        ├── hg38_gnomad40_genome.txt
        ├── hg38_clinvar_20240917.txt
        ├── hg38_dbnsfp42a.txt
        └── hg38_cosmic70.txt
```

In [0]:
# ── Widget Parameters ──────────────────────────────────────────────────────────
dbutils.widgets.text("reference_volume",
    "/Volumes/dhbl_discovery_us_dev/genesis_schema/variant_annotation_reference",
    "Reference Volume")
dbutils.widgets.dropdown("genome_build", "GRCh38", ["GRCh38", "GRCh37"], "Genome Build")
dbutils.widgets.dropdown("install_components", "all",
    ["all", "reference_only", "vep_only", "annovar_only"], "Components to Install")
dbutils.widgets.text("annovar_download_url", "",
    "ANNOVAR Download URL (from registration)")
dbutils.widgets.text("annovar_protocols",
    "refGene,gnomad40_genome,clinvar_20240917,dbnsfp42a,cosmic70",
    "ANNOVAR Protocols")
dbutils.widgets.text("annovar_operations", "g,f,f,f,f", "ANNOVAR Operations")
dbutils.widgets.dropdown("install_cadd_plugin", "no", ["yes", "no"],
    "Install CADD Plugin (~30 GB extra)")

# ── Read all parameters ──
reference_volume   = dbutils.widgets.get("reference_volume")
genome_build       = dbutils.widgets.get("genome_build")
install_components = dbutils.widgets.get("install_components")
annovar_download_url = dbutils.widgets.get("annovar_download_url")
annovar_protocols  = dbutils.widgets.get("annovar_protocols")
annovar_operations = dbutils.widgets.get("annovar_operations")
install_cadd       = dbutils.widgets.get("install_cadd_plugin") == "yes"

# Derived flags
install_ref    = install_components in ("all", "reference_only")
install_vep    = install_components in ("all", "vep_only")
install_annovar = install_components in ("all", "annovar_only")

# ANNOVAR build version mapping
annovar_buildver = "hg38" if genome_build == "GRCh38" else "hg19"

print(f"reference_volume:     {reference_volume}")
print(f"genome_build:         {genome_build}")
print(f"install_components:   {install_components}")
print(f"annovar_download_url: {'(provided)' if annovar_download_url else '(not set)'}")
print(f"annovar_protocols:    {annovar_protocols}")
print(f"annovar_operations:   {annovar_operations}")
print(f"annovar_buildver:     {annovar_buildver}")
print(f"install_cadd_plugin:  {install_cadd}")
print(f"\nComponents to install:")
print(f"  Reference genome:   {'yes' if install_ref else 'skip'}")
print(f"  Ensembl VEP 115:    {'yes' if install_vep else 'skip'}")
print(f"  ANNOVAR:            {'yes' if install_annovar else 'skip'}")
print(f"  CADD plugin:        {'yes' if install_cadd else 'skip'}")

In [0]:
import os, subprocess, time, shutil

# ── Define all paths ──
ref_dir       = os.path.join(reference_volume, "reference")
vep_cache_dir = os.path.join(reference_volume, "vep_cache")
vep_bin_dir   = os.path.join(vep_cache_dir, "ensembl-vep-release-115")
annovar_dir   = os.path.join(reference_volume, "annovar")
humandb_dir   = os.path.join(annovar_dir, "humandb")
cadd_dir      = os.path.join(vep_cache_dir, "CADD")

# Genome-specific
ref_fasta     = os.path.join(ref_dir, f"{genome_build}.fa")
ref_fai       = os.path.join(ref_dir, f"{genome_build}.fa.fai")
vep_species   = os.path.join(vep_cache_dir, "homo_sapiens")
cache_version = "115_GRCh38" if genome_build == "GRCh38" else "115_GRCh37"
vep_cache_sub = os.path.join(vep_species, cache_version)

# ── Create directories ──
for d in [ref_dir, vep_cache_dir, annovar_dir, humandb_dir]:
    os.makedirs(d, exist_ok=True)
    print(f"  \u2705 {d}")

# ── Report existing contents ──
print(f"\nExisting contents of {reference_volume}:")
for root, dirs, files in os.walk(reference_volume):
    depth = root.replace(reference_volume, "").count(os.sep)
    indent = "  " * (depth + 1)
    subindent = "  " * (depth + 2)
    dirname = os.path.basename(root) or reference_volume.split("/")[-1]
    if depth <= 2:
        dir_size = sum(
            os.path.getsize(os.path.join(root, f))
            for f in files
        )
        size_str = f" ({dir_size / (1024**3):.1f} GB)" if dir_size > 0 else " (empty)"
        print(f"{indent}{dirname}/{size_str}")
    if depth > 2:
        break

## Stage 1 — Reference Genome FASTA

VEP requires a reference FASTA for HGVS notation and variant consequence prediction. The FASTA must be indexed with `samtools faidx`. Download source: **Ensembl FTP** (release 115).

In [0]:
# ── Download and index reference genome FASTA ─────────────────────
if not install_ref and not install_vep:
    print("\u23ed\ufe0f  Reference genome download skipped")
else:
    if os.path.isfile(ref_fasta) and os.path.getsize(ref_fasta) > 1_000_000_000:
        size_gb = os.path.getsize(ref_fasta) / (1024**3)
        print(f"\u2705 Reference FASTA already exists: {ref_fasta} ({size_gb:.1f} GB)")
    else:
        # Ensembl FTP URLs
        if genome_build == "GRCh38":
            fasta_url = "https://ftp.ensembl.org/pub/release-115/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz"
        else:
            fasta_url = "https://ftp.ensembl.org/pub/grch37/release-115/fasta/homo_sapiens/dna/Homo_sapiens.GRCh37.dna.primary_assembly.fa.gz"

        tmp_gz = f"/tmp/{genome_build}_primary_assembly.fa.gz"
        tmp_fa = f"/tmp/{genome_build}_primary_assembly.fa"

        print(f"\u25b6 Downloading reference genome ({genome_build}) ...")
        print(f"  URL: {fasta_url}")
        print(f"  Downloading to /tmp first for speed ...")

        t0 = time.time()
        proc = subprocess.run(
            ["wget", "-q", "--show-progress", "-O", tmp_gz, fasta_url],
            capture_output=True, text=True, timeout=3600
        )
        if proc.returncode != 0:
            # Try curl as fallback
            print("  wget failed, trying curl ...")
            proc = subprocess.run(
                ["curl", "-L", "--fail", "-o", tmp_gz, fasta_url],
                capture_output=True, text=True, timeout=3600
            )
        if proc.returncode != 0:
            raise RuntimeError(f"Download failed: {proc.stderr[-500:]}")

        gz_size = os.path.getsize(tmp_gz) / (1024**3)
        print(f"  \u2705 Downloaded {gz_size:.1f} GB in {time.time() - t0:.0f}s")

        # Validate downloaded file is actually gzip
        with open(tmp_gz, "rb") as f:
            magic = f.read(2)
        if magic != b'\x1f\x8b':
            with open(tmp_gz, "r", errors="replace") as f:
                head = f.read(500)
            os.remove(tmp_gz)
            raise RuntimeError(
                f"Downloaded file is not gzip format. The URL may be invalid or blocked.\n"
                f"URL: {fasta_url}\nFile starts with: {head[:200]}"
            )

        # Decompress
        print("  Decompressing ...")
        subprocess.run(["gunzip", "-f", tmp_gz], check=True, timeout=600)
        fa_size = os.path.getsize(tmp_fa) / (1024**3)
        print(f"  \u2705 Decompressed: {fa_size:.1f} GB")

        # Copy to Volume
        print(f"  Copying to {ref_fasta} ...")
        shutil.copy2(tmp_fa, ref_fasta)
        os.remove(tmp_fa)
        print(f"  \u2705 Reference genome installed")

    # ── Index with samtools ──
    if os.path.isfile(ref_fai):
        print(f"\u2705 FASTA index already exists: {ref_fai}")
    else:
        print("\u25b6 Indexing with samtools faidx ...")
        # Ensure samtools is available
        if not shutil.which("samtools"):
            print("  Installing samtools via apt-get ...")
            subprocess.run(
                ["sudo", "apt-get", "install", "-y", "-qq", "samtools"],
                capture_output=True, timeout=120
            )

        if shutil.which("samtools"):
            proc = subprocess.run(
                ["samtools", "faidx", ref_fasta],
                capture_output=True, text=True, timeout=600
            )
            if proc.returncode == 0:
                print(f"  \u2705 Index created: {ref_fai}")
            else:
                print(f"  \u26a0\ufe0f samtools faidx failed: {proc.stderr[:300]}")
                print("  VEP can still run without .fai but HGVS notation may be slower")
        else:
            print("  \u26a0\ufe0f samtools not available. VEP will still work but HGVS may be slower.")

    # Summary
    print(f"\nReference genome summary:")
    for f in [ref_fasta, ref_fai]:
        if os.path.isfile(f):
            s = os.path.getsize(f)
            unit = "GB" if s > 1e9 else "MB" if s > 1e6 else "KB"
            val = s / (1024**3) if s > 1e9 else s / (1024**2) if s > 1e6 else s / 1024
            print(f"  {os.path.basename(f):30s}  {val:.1f} {unit}")
        else:
            print(f"  {os.path.basename(f):30s}  MISSING")

## Stage 2 — Ensembl VEP 115

Two steps:
1. **Clone VEP source** from GitHub (release/115 branch) — provides the `vep` Perl script
2. **Download pre-built cache** from Ensembl FTP — \~17 GB of pre-indexed variant annotations

Using the pre-built cache is much faster than running `INSTALL.pl` (which compiles from source).

In [0]:
# ── Clone Ensembl VEP release/115 source ──────────────────────────────────────
if not install_vep:
    print("\u23ed\ufe0f  VEP source clone skipped")
else:
    vep_script = os.path.join(vep_bin_dir, "vep")

    # Ensure PERL5LIB includes VEP and ensembl API paths
    ensembl_api_modules = os.path.join(vep_bin_dir, "ensembl", "modules")
    perl5lib_paths = [vep_bin_dir, ensembl_api_modules]
    existing_perl5lib = os.environ.get("PERL5LIB", "")
    if existing_perl5lib:
        perl5lib_paths.append(existing_perl5lib)
    os.environ["PERL5LIB"] = ":".join(perl5lib_paths)

    if os.path.isfile(vep_script):
        print(f"\u2705 VEP source already cloned at {vep_bin_dir}")
        # Show version
        proc = subprocess.run(
            ["perl", vep_script, "--help"],
            capture_output=True, text=True, timeout=30
        )
        for line in proc.stdout.split("\n")[:3]:
            if line.strip():
                print(f"  {line.strip()}")
    else:
        print("\u25b6 Cloning Ensembl VEP release/115 from GitHub ...")

        # Clone to /tmp first (faster), then move to Volume
        tmp_clone = "/tmp/ensembl-vep-release-115"
        if os.path.isdir(tmp_clone):
            shutil.rmtree(tmp_clone)

        t0 = time.time()
        proc = subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "release/115",
             "https://github.com/Ensembl/ensembl-vep.git", tmp_clone],
            capture_output=True, text=True, timeout=300
        )
        if proc.returncode != 0:
            raise RuntimeError(f"git clone failed: {proc.stderr[-500:]}")

        elapsed = time.time() - t0
        print(f"  \u2705 Cloned in {elapsed:.0f}s")

        # Move to Volume
        print(f"  Copying to {vep_bin_dir} ...")
        if os.path.isdir(vep_bin_dir):
            shutil.rmtree(vep_bin_dir)
        shutil.copytree(tmp_clone, vep_bin_dir)
        shutil.rmtree(tmp_clone)

        # Verify
        if os.path.isfile(vep_script):
            print(f"  \u2705 VEP script verified: {vep_script}")
        else:
            raise RuntimeError(f"VEP script not found after clone: {vep_script}")

    # ── Install Ensembl Perl API dependencies ──
    ensembl_api_dir = os.path.join(vep_bin_dir, "ensembl")
    if os.path.isdir(os.path.join(ensembl_api_dir, "modules", "Bio")):
        print(f"\u2705 Ensembl Perl API already installed at {ensembl_api_dir}")
    else:
        print("\n\u25b6 Installing Ensembl Perl API (required for Bio::EnsEMBL modules) ...")
        t0 = time.time()
        proc = subprocess.run(
            ["perl", "INSTALL.pl", "--AUTO", "a", "--NO_HTSLIB", "--NO_UPDATE"],
            capture_output=True, text=True, timeout=600,
            cwd=vep_bin_dir
        )
        if proc.returncode != 0:
            print(f"  \u26a0\ufe0f INSTALL.pl returned code {proc.returncode}")
            print(f"  stderr: {proc.stderr[-300:]}")
            print("  Trying alternative: cloning ensembl core API directly ...")
            # Fallback: clone ensembl core repo for the Perl modules
            # Must clone to /tmp first — UC Volumes don't support git pack operations
            tmp_ensembl = "/tmp/ensembl-api-115"
            if os.path.isdir(tmp_ensembl):
                shutil.rmtree(tmp_ensembl)
            proc2 = subprocess.run(
                ["git", "clone", "--depth", "1", "--branch", "release/115",
                 "https://github.com/Ensembl/ensembl.git", tmp_ensembl],
                capture_output=True, text=True, timeout=300
            )
            if proc2.returncode != 0:
                raise RuntimeError(f"Failed to install Ensembl API: {proc2.stderr[-300:]}")
            # Copy to Volume
            if os.path.isdir(ensembl_api_dir):
                shutil.rmtree(ensembl_api_dir)
            shutil.copytree(tmp_ensembl, ensembl_api_dir)
            shutil.rmtree(tmp_ensembl)
            print(f"  \u2705 Ensembl core API cloned to {ensembl_api_dir}")
        else:
            elapsed = time.time() - t0
            print(f"  \u2705 Ensembl API installed in {elapsed:.0f}s")

    # ── Check Perl dependencies ──
    print("\nChecking Perl dependencies ...")
    perl_modules = [
        ("Bio::EnsEMBL::Registry", "required for VEP core"),
        ("DBI", "required for cache"),
        ("Archive::Zip", "required for cache extraction"),
        ("JSON", "optional, for JSON output"),
        ("Set::IntervalTree", "optional, for speed"),
        ("Bio::DB::HTS", "optional, for BAM/CRAM support"),
    ]
    for mod, desc in perl_modules:
        proc = subprocess.run(
            ["perl", "-e", f"use {mod}; print 'OK';"],
            capture_output=True, text=True, timeout=10
        )
        icon = "\u2705" if proc.returncode == 0 else "\u26a0\ufe0f"
        print(f"  {icon}  {mod:25s} ({desc})")

In [0]:
# ── Download pre-built VEP cache from Ensembl FTP ─────────────────────────
if not install_vep:
    print("\u23ed\ufe0f  VEP cache download skipped")
else:
    # Check if cache already exists
    if os.path.isdir(vep_cache_sub) and len(os.listdir(vep_cache_sub)) > 5:
        cache_size = sum(
            os.path.getsize(os.path.join(dp, f))
            for dp, _, fn in os.walk(vep_cache_sub)
            for f in fn
        ) / (1024**3)
        file_count = sum(len(fn) for _, _, fn in os.walk(vep_cache_sub))
        print(f"\u2705 VEP cache already exists: {vep_cache_sub}")
        print(f"   {file_count} files, {cache_size:.1f} GB")
    else:
        # Build download URL
        if genome_build == "GRCh38":
            cache_url = "https://ftp.ensembl.org/pub/release-115/variation/indexed_vep_cache/homo_sapiens_vep_115_GRCh38.tar.gz"
        else:
            cache_url = "https://ftp.ensembl.org/pub/release-115/variation/indexed_vep_cache/homo_sapiens_vep_115_GRCh37.tar.gz"

        cache_tarball = os.path.basename(cache_url)
        tmp_tarball = f"/tmp/{cache_tarball}"

        print(f"\u25b6 Downloading VEP {genome_build} cache (this is ~17 GB, may take 20-40 min) ...")
        print(f"  URL: {cache_url}")

        # Download with progress monitoring
        t0 = time.time()

        # Use wget with continue support in case of interruption
        proc = subprocess.run(
            ["wget", "-c", "-q", "--show-progress", "-O", tmp_tarball, cache_url],
            capture_output=True, text=True, timeout=7200  # 2 hour timeout
        )
        if proc.returncode != 0:
            print("  wget failed, trying curl ...")
            proc = subprocess.run(
                ["curl", "-L", "-C", "-", "--fail", "-o", tmp_tarball, cache_url],
                capture_output=True, text=True, timeout=7200
            )
        if proc.returncode != 0:
            raise RuntimeError(f"Cache download failed: {proc.stderr[-500:]}")

        dl_size = os.path.getsize(tmp_tarball) / (1024**3)
        dl_time = time.time() - t0
        print(f"  \u2705 Downloaded {dl_size:.1f} GB in {dl_time:.0f}s ({dl_size * 1024 / dl_time:.1f} MB/s)")

        # Validate downloaded file is actually gzip before attempting extraction
        with open(tmp_tarball, "rb") as f:
            magic = f.read(2)
        if magic != b'\x1f\x8b':
            with open(tmp_tarball, "r", errors="replace") as f:
                head = f.read(500)
            os.remove(tmp_tarball)
            raise RuntimeError(
                f"Downloaded file is not gzip format. The URL may be invalid or blocked.\n"
                f"URL: {cache_url}\nFile starts with: {head[:200]}"
            )

        # Extract to vep_cache/ — tarball contains homo_sapiens/115_GRCh38/ structure
        print(f"  Extracting to {vep_cache_dir} (this may take 5-10 min) ...")
        t0 = time.time()
        proc = subprocess.run(
            ["tar", "xzf", tmp_tarball, "-C", vep_cache_dir],
            capture_output=True, text=True, timeout=3600
        )
        if proc.returncode != 0:
            raise RuntimeError(f"Extraction failed: {proc.stderr[-500:]}")

        extract_time = time.time() - t0
        print(f"  \u2705 Extracted in {extract_time:.0f}s")

        # Clean up tarball from /tmp
        os.remove(tmp_tarball)
        print(f"  \u2705 Cleaned up {tmp_tarball}")

        # Report cache contents
        if os.path.isdir(vep_cache_sub):
            cache_size = sum(
                os.path.getsize(os.path.join(dp, f))
                for dp, _, fn in os.walk(vep_cache_sub)
                for f in fn
            ) / (1024**3)
            file_count = sum(len(fn) for _, _, fn in os.walk(vep_cache_sub))
            print(f"\n  Cache installed: {vep_cache_sub}")
            print(f"  {file_count} files, {cache_size:.1f} GB")
        else:
            raise RuntimeError(f"Cache directory not found after extraction: {vep_cache_sub}")

In [0]:
# ── Verify VEP 115 installation ─────────────────────────────────────────────
if not install_vep:
    print("\u23ed\ufe0f  VEP verification skipped")
else:
    print("VEP 115 Verification")
    print("=" * 50)

    checks = {
        "VEP Perl script":          os.path.isfile(os.path.join(vep_bin_dir, "vep")),
        "VEP modules directory":    os.path.isdir(os.path.join(vep_bin_dir, "modules")),
        "homo_sapiens cache dir":   os.path.isdir(vep_species),
        f"{cache_version} cache":   os.path.isdir(vep_cache_sub),
        "Reference FASTA":          os.path.isfile(ref_fasta),
        "Reference FASTA index":    os.path.isfile(ref_fai),
    }

    all_ok = True
    for name, passed in checks.items():
        icon = "\u2705" if passed else "\u274c"
        print(f"  {icon}  {name}")
        if not passed:
            all_ok = False

    # Quick functional test — run vep --help
    vep_script = os.path.join(vep_bin_dir, "vep")
    if os.path.isfile(vep_script):
        proc = subprocess.run(
            ["perl", vep_script, "--help"],
            capture_output=True, text=True, timeout=30
        )
        if proc.returncode == 0:
            # Extract version line
            for line in proc.stdout.split("\n"):
                if "ensembl-vep" in line.lower() or "version" in line.lower():
                    print(f"\n  Version: {line.strip()}")
                    break
            print("  \u2705 VEP --help executed successfully")
        else:
            print(f"  \u26a0\ufe0f  VEP --help returned exit code {proc.returncode}")
            if proc.stderr:
                print(f"     {proc.stderr[:200]}")

    print("=" * 50)
    status = "\u2705 VEP 115 READY" if all_ok else "\u26a0\ufe0f VEP 115 INCOMPLETE"
    print(status)

## Stage 3 — ANNOVAR

ANNOVAR requires registration. Visit https://annovar.openbioinformatics.org/en/latest/user-guide/download/ to register and get a personalized download URL.

**Two installation options:**
- **Option A**: Provide your download URL in the `annovar_download_url` widget above
- **Option B**: Manually upload `annovar.latest.tar.gz` to the reference volume (`{reference_volume}/`) and this notebook will extract it

After extracting ANNOVAR, we use `annotate_variation.pl -downdb` to download each annotation database.

In [0]:
install_annovar = False  # Stage 3 commented out — re-enable when ANNOVAR download URL is available

# ── Install ANNOVAR from download URL or pre-uploaded tarball ─────────────────
if not install_annovar:
    print("\u23ed\ufe0f  ANNOVAR installation skipped")
else:
    annovar_script = os.path.join(annovar_dir, "table_annovar.pl")

    if os.path.isfile(annovar_script):
        print(f"\u2705 ANNOVAR already installed at {annovar_dir}")
        # List available scripts
        scripts = [f for f in os.listdir(annovar_dir) if f.endswith(".pl")]
        print(f"  Scripts: {', '.join(sorted(scripts)[:5])} ...")
    else:
        # Option A: Download from URL
        # Option B: Use pre-uploaded tarball on Volume
        tarball_path = None
        tmp_tarball = "/tmp/annovar.latest.tar.gz"

        if annovar_download_url.strip():
            print("\u25b6 Downloading ANNOVAR from provided URL (Option A) ...")
            t0 = time.time()
            proc = subprocess.run(
                ["wget", "-q", "--show-progress", "-O", tmp_tarball,
                 annovar_download_url.strip()],
                capture_output=True, text=True, timeout=1800
            )
            if proc.returncode != 0:
                proc = subprocess.run(
                    ["curl", "-L", "-o", tmp_tarball, annovar_download_url.strip()],
                    capture_output=True, text=True, timeout=1800
                )
            if proc.returncode != 0:
                raise RuntimeError(
                    f"ANNOVAR download failed. Verify your URL is correct.\n{proc.stderr[-300:]}"
                )
            tarball_path = tmp_tarball
            dl_size = os.path.getsize(tarball_path) / (1024**2)
            print(f"  \u2705 Downloaded {dl_size:.0f} MB in {time.time() - t0:.0f}s")

        else:
            # Check for pre-uploaded tarball on Volume
            volume_tarball = os.path.join(reference_volume, "annovar.latest.tar.gz")
            if os.path.isfile(volume_tarball):
                print(f"\u25b6 Using pre-uploaded tarball (Option B): {volume_tarball}")
                # Copy to /tmp for faster extraction
                shutil.copy2(volume_tarball, tmp_tarball)
                tarball_path = tmp_tarball
            else:
                raise RuntimeError(
                    "ANNOVAR not installed and no download source found.\n"
                    "Either:\n"
                    "  (A) Set 'annovar_download_url' widget to your registered URL, or\n"
                    f"  (B) Upload annovar.latest.tar.gz to {reference_volume}/\n"
                    "Register at: https://annovar.openbioinformatics.org/en/latest/user-guide/download/"
                )

        # Extract ANNOVAR tarball
        print(f"  Extracting ANNOVAR ...")
        # ANNOVAR tarball typically contains an 'annovar/' directory
        proc = subprocess.run(
            ["tar", "xzf", tarball_path, "-C", "/tmp/"],
            capture_output=True, text=True, timeout=120
        )
        if proc.returncode != 0:
            raise RuntimeError(f"Extraction failed: {proc.stderr}")

        # Find the extracted directory (usually /tmp/annovar/)
        extracted_dir = "/tmp/annovar"
        if not os.path.isdir(extracted_dir):
            # Try to find it
            candidates = [d for d in os.listdir("/tmp")
                          if d.startswith("annovar") and os.path.isdir(f"/tmp/{d}")]
            if candidates:
                extracted_dir = f"/tmp/{candidates[0]}"
            else:
                raise RuntimeError("Could not find extracted ANNOVAR directory")

        # Copy Perl scripts and modules to Volume
        print(f"  Copying to {annovar_dir} ...")
        for item in os.listdir(extracted_dir):
            src = os.path.join(extracted_dir, item)
            dst = os.path.join(annovar_dir, item)
            if os.path.isfile(src):
                shutil.copy2(src, dst)
            elif os.path.isdir(src) and item != "humandb":
                if os.path.isdir(dst):
                    shutil.rmtree(dst)
                shutil.copytree(src, dst)

        # Create humandb if it doesn't exist
        os.makedirs(humandb_dir, exist_ok=True)

        # Clean up
        shutil.rmtree(extracted_dir, ignore_errors=True)
        if os.path.isfile(tmp_tarball):
            os.remove(tmp_tarball)

        # Verify
        required_scripts = ["annotate_variation.pl", "table_annovar.pl", "convert2annovar.pl"]
        for script in required_scripts:
            path = os.path.join(annovar_dir, script)
            icon = "\u2705" if os.path.isfile(path) else "\u274c"
            print(f"  {icon}  {script}")

        if not os.path.isfile(annovar_script):
            raise RuntimeError("ANNOVAR installation failed — table_annovar.pl not found")

        print(f"\n\u2705 ANNOVAR installed successfully")

In [0]:
# ── Download ANNOVAR annotation databases ──────────────────────────────────
if not install_annovar:
    print("\u23ed\ufe0f  ANNOVAR database download skipped")
else:
    annotate_pl = os.path.join(annovar_dir, "annotate_variation.pl")
    if not os.path.isfile(annotate_pl):
        raise RuntimeError(f"annotate_variation.pl not found at {annotate_pl}")

    protocols = [p.strip() for p in annovar_protocols.split(",")]
    operations = [o.strip() for o in annovar_operations.split(",")]

    if len(protocols) != len(operations):
        raise ValueError(
            f"Protocol count ({len(protocols)}) != operation count ({len(operations)}).\n"
            f"Protocols:  {protocols}\n"
            f"Operations: {operations}"
        )

    print(f"\u25b6 Downloading {len(protocols)} ANNOVAR databases for {annovar_buildver} ...")
    print(f"  Target: {humandb_dir}\n")

    results = []
    for i, (protocol, op) in enumerate(zip(protocols, operations), 1):
        # Check if database already exists
        # Gene-based: {buildver}_{protocol}.txt
        # Filter-based: {buildver}_{protocol}.txt
        expected_file = os.path.join(humandb_dir, f"{annovar_buildver}_{protocol}.txt")
        # Some databases use different naming patterns
        alt_patterns = [
            f"{annovar_buildver}_{protocol}.txt",
            f"{annovar_buildver}_{protocol}.txt.gz",
            f"{annovar_buildver}_{protocol}.txt.idx",
        ]
        already_exists = any(
            os.path.isfile(os.path.join(humandb_dir, p)) for p in alt_patterns
        )

        if already_exists:
            size = 0
            for p in alt_patterns:
                fp = os.path.join(humandb_dir, p)
                if os.path.isfile(fp):
                    size = os.path.getsize(fp)
            size_str = f"{size / (1024**3):.1f} GB" if size > 1e9 else f"{size / (1024**2):.0f} MB"
            print(f"  [{i}/{len(protocols)}] \u2705 {protocol} already installed ({size_str})")
            results.append((protocol, "exists", size_str))
            continue

        print(f"  [{i}/{len(protocols)}] \u25b6 Downloading {protocol} ({op}) ...")
        t0 = time.time()

        cmd = [
            "perl", annotate_pl,
            "-buildver", annovar_buildver,
            "-downdb", "-webfrom", "annovar",
            protocol,
            humandb_dir,
        ]

        proc = subprocess.run(
            cmd, capture_output=True, text=True,
            timeout=7200  # 2 hour timeout per database
        )

        elapsed = time.time() - t0

        if proc.returncode != 0:
            print(f"           \u26a0\ufe0f  {protocol} download failed (exit {proc.returncode})")
            if proc.stderr:
                # Show last few lines of error
                err_lines = proc.stderr.strip().split("\n")[-3:]
                for line in err_lines:
                    print(f"           {line}")
            results.append((protocol, "FAILED", "-"))
            continue

        # Check result
        db_files = [f for f in os.listdir(humandb_dir)
                    if f.startswith(f"{annovar_buildver}_{protocol}")]
        if db_files:
            total_size = sum(
                os.path.getsize(os.path.join(humandb_dir, f)) for f in db_files
            )
            size_str = f"{total_size / (1024**3):.1f} GB" if total_size > 1e9 else f"{total_size / (1024**2):.0f} MB"
            print(f"           \u2705 Installed in {elapsed:.0f}s ({size_str}, {len(db_files)} files)")
            results.append((protocol, "installed", size_str))
        else:
            print(f"           \u26a0\ufe0f  No files found after download")
            results.append((protocol, "unknown", "-"))

    # ── Summary table ──
    print(f"\n{'=' * 55}")
    print(f"  {'Protocol':<25s} {'Status':<12s} {'Size':<12s}")
    print(f"  {'-' * 25} {'-' * 12} {'-' * 12}")
    for protocol, status, size in results:
        icon = "\u2705" if status in ("exists", "installed") else "\u274c"
        print(f"  {icon} {protocol:<23s} {status:<12s} {size:<12s}")
    print(f"{'=' * 55}")

    failed = [r[0] for r in results if r[1] == "FAILED"]
    if failed:
        print(f"\n\u26a0\ufe0f  {len(failed)} database(s) failed: {', '.join(failed)}")
        print("  Re-run this cell to retry failed downloads.")
    else:
        print(f"\n\u2705 All {len(results)} databases ready")

In [0]:
# ── Verify complete ANNOVAR installation ─────────────────────────────────────
if not install_annovar:
    print("\u23ed\ufe0f  ANNOVAR verification skipped")
else:
    print("ANNOVAR Verification")
    print("=" * 50)

    # Check scripts
    required_scripts = [
        "annotate_variation.pl",
        "table_annovar.pl",
        "convert2annovar.pl",
        "coding_change.pl",
        "retrieve_seq_from_fasta.pl",
    ]
    for script in required_scripts:
        path = os.path.join(annovar_dir, script)
        icon = "\u2705" if os.path.isfile(path) else "\u274c"
        print(f"  {icon}  {script}")

    # Check databases
    print(f"\n  Databases in {humandb_dir}:")
    protocols = [p.strip() for p in annovar_protocols.split(",")]
    all_dbs_ok = True
    for protocol in protocols:
        db_files = [f for f in os.listdir(humandb_dir)
                    if f.startswith(f"{annovar_buildver}_{protocol}")] if os.path.isdir(humandb_dir) else []
        if db_files:
            total_size = sum(
                os.path.getsize(os.path.join(humandb_dir, f)) for f in db_files
            )
            size_str = f"{total_size / (1024**3):.1f} GB" if total_size > 1e9 else f"{total_size / (1024**2):.0f} MB"
            print(f"    \u2705 {protocol:30s} {size_str:>10s}  ({len(db_files)} files)")
        else:
            print(f"    \u274c {protocol:30s} MISSING")
            all_dbs_ok = False

    # Quick functional test
    print()
    annovar_script = os.path.join(annovar_dir, "table_annovar.pl")
    if os.path.isfile(annovar_script):
        proc = subprocess.run(
            ["perl", annovar_script],
            capture_output=True, text=True, timeout=10
        )
        # table_annovar.pl with no args prints usage and exits non-zero, but that's expected
        if "table_annovar.pl" in (proc.stdout + proc.stderr).lower():
            print("  \u2705 table_annovar.pl executes correctly")
        else:
            print("  \u26a0\ufe0f  Unexpected output from table_annovar.pl")

    print("=" * 50)
    if all_dbs_ok:
        print("\u2705 ANNOVAR READY")
    else:
        print("\u26a0\ufe0f ANNOVAR INCOMPLETE — some databases missing")

## Stage 4 — Optional VEP Plugins (CADD)

CADD (Combined Annotation Dependent Depletion) scores provide a single measure of variant deleteriousness. The plugin data is large:
- `whole_genome_SNVs.tsv.gz` \~20 GB (all possible SNVs)
- `gnomad.genomes.r4.0.indel.tsv.gz` \~8 GB (observed indels)

Only installed if `install_cadd_plugin = yes`. Source: https://cadd.gs.washington.edu/download

In [0]:
# ── Download CADD plugin data (optional, ~30 GB) ───────────────────────
if not install_cadd:
    print("\u23ed\ufe0f  CADD plugin download skipped (set install_cadd_plugin=yes to enable)")
else:
    os.makedirs(cadd_dir, exist_ok=True)

    cadd_files = {
        "whole_genome_SNVs.tsv.gz": {
            "url": "https://krishna.gs.washington.edu/download/CADD/v1.7/GRCh38/whole_genome_SNVs.tsv.gz",
            "desc": "All possible SNVs (~20 GB)",
        },
        "whole_genome_SNVs.tsv.gz.tbi": {
            "url": "https://krishna.gs.washington.edu/download/CADD/v1.7/GRCh38/whole_genome_SNVs.tsv.gz.tbi",
            "desc": "SNV tabix index",
        },
        "gnomad.genomes.r4.0.indel.tsv.gz": {
            "url": "https://krishna.gs.washington.edu/download/CADD/v1.7/GRCh38/gnomad.genomes.r4.0.indel.tsv.gz",
            "desc": "gnomAD indels (~8 GB)",
        },
        "gnomad.genomes.r4.0.indel.tsv.gz.tbi": {
            "url": "https://krishna.gs.washington.edu/download/CADD/v1.7/GRCh38/gnomad.genomes.r4.0.indel.tsv.gz.tbi",
            "desc": "Indel tabix index",
        },
    }

    if genome_build == "GRCh37":
        print("\u26a0\ufe0f  CADD GRCh37 URLs differ. Adjust URLs manually for hg19.")
        print("   Skipping CADD download for GRCh37.")
    else:
        print(f"\u25b6 Downloading CADD v1.7 plugin data to {cadd_dir} ...")
        print(f"  This is ~30 GB total and may take 30-60+ minutes.\n")

        for filename, info in cadd_files.items():
            dest = os.path.join(cadd_dir, filename)

            if os.path.isfile(dest) and os.path.getsize(dest) > 1000:
                size = os.path.getsize(dest)
                size_str = f"{size / (1024**3):.1f} GB" if size > 1e9 else f"{size / (1024**2):.0f} MB"
                print(f"  \u2705 {filename} already exists ({size_str})")
                continue

            print(f"  \u25b6 {filename} — {info['desc']}")
            tmp_dest = f"/tmp/{filename}"

            t0 = time.time()
            proc = subprocess.run(
                ["wget", "-c", "-q", "--show-progress", "-O", tmp_dest, info["url"]],
                capture_output=True, text=True, timeout=14400  # 4 hour timeout
            )
            if proc.returncode != 0:
                proc = subprocess.run(
                    ["curl", "-L", "-C", "-", "-o", tmp_dest, info["url"]],
                    capture_output=True, text=True, timeout=14400
                )
            if proc.returncode != 0:
                print(f"    \u274c Download failed: {proc.stderr[-200:]}")
                continue

            elapsed = time.time() - t0
            size = os.path.getsize(tmp_dest)
            size_str = f"{size / (1024**3):.1f} GB" if size > 1e9 else f"{size / (1024**2):.0f} MB"
            print(f"    Downloaded {size_str} in {elapsed:.0f}s")

            # Copy to Volume
            print(f"    Copying to Volume ...")
            shutil.copy2(tmp_dest, dest)
            os.remove(tmp_dest)
            print(f"    \u2705 Installed")

        print(f"\n\u2705 CADD plugin data ready at {cadd_dir}")

In [0]:
# ── Final validation summary ──────────────────────────────────────────────────
def file_size_str(path):
    """Return a human-readable file size, or 'MISSING'."""
    if not os.path.isfile(path):
        return "MISSING"
    s = os.path.getsize(path)
    if s > 1e9:
        return f"{s / (1024**3):.1f} GB"
    elif s > 1e6:
        return f"{s / (1024**2):.0f} MB"
    return f"{s / 1024:.0f} KB"

def dir_size_gb(path):
    """Return total size of directory in GB."""
    if not os.path.isdir(path):
        return 0.0
    total = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, fn in os.walk(path)
        for f in fn
    )
    return total / (1024**3)

# Collect all checks
ref_ok      = os.path.isfile(ref_fasta) and os.path.getsize(ref_fasta) > 1e9
ref_idx_ok  = os.path.isfile(ref_fai)
vep_src_ok  = os.path.isfile(os.path.join(vep_bin_dir, "vep"))
vep_cache_ok = os.path.isdir(vep_cache_sub) and len(os.listdir(vep_cache_sub)) > 5
annovar_ok  = os.path.isfile(os.path.join(annovar_dir, "table_annovar.pl"))
cadd_ok     = os.path.isfile(os.path.join(cadd_dir, "whole_genome_SNVs.tsv.gz")) if install_cadd else None

# Count ANNOVAR databases
protocols = [p.strip() for p in annovar_protocols.split(",")]
db_count = 0
for protocol in protocols:
    if any(
        f.startswith(f"{annovar_buildver}_{protocol}")
        for f in os.listdir(humandb_dir)
    ) if os.path.isdir(humandb_dir) else False:
        db_count += 1

# Total disk usage
total_gb = dir_size_gb(reference_volume)

# ── Print summary box ──
w = 66
yn = lambda x: "yes" if x else "NO"

print()
print("\u2554" + "\u2550" * w + "\u2557")
print("\u2551" + " REFERENCE DATABASE SETUP COMPLETE".center(w) + "\u2551")
print("\u2560" + "\u2550" * w + "\u2563")
print("\u2551" + f"  Volume: {reference_volume}".ljust(w) + "\u2551")
print("\u2551" + f"  Genome build: {genome_build}".ljust(w) + "\u2551")
print("\u2551" + f"  Total disk usage: {total_gb:.1f} GB".ljust(w) + "\u2551")
print("\u2560" + "\u2550" * w + "\u2563")
print("\u2551" + f"  Reference Genome".ljust(w) + "\u2551")
print("\u2551" + f"    FASTA:    {yn(ref_ok):4s}  {file_size_str(ref_fasta)}".ljust(w) + "\u2551")
print("\u2551" + f"    Index:    {yn(ref_idx_ok):4s}  {file_size_str(ref_fai)}".ljust(w) + "\u2551")
print("\u2560" + "\u2550" * w + "\u2563")
print("\u2551" + f"  Ensembl VEP 115".ljust(w) + "\u2551")
print("\u2551" + f"    Source:   {yn(vep_src_ok):4s}  {vep_bin_dir}".ljust(w) + "\u2551")
print("\u2551" + f"    Cache:    {yn(vep_cache_ok):4s}  {dir_size_gb(vep_cache_sub):.1f} GB".ljust(w) + "\u2551")
print("\u2560" + "\u2550" * w + "\u2563")
print("\u2551" + f"  ANNOVAR".ljust(w) + "\u2551")
print("\u2551" + f"    Scripts:  {yn(annovar_ok):4s}".ljust(w) + "\u2551")
print("\u2551" + f"    Databases: {db_count}/{len(protocols)} installed ({dir_size_gb(humandb_dir):.1f} GB)".ljust(w) + "\u2551")
if install_cadd:
    print("\u2560" + "\u2550" * w + "\u2563")
    print("\u2551" + f"  CADD Plugin".ljust(w) + "\u2551")
    print("\u2551" + f"    Data:     {yn(cadd_ok):4s}  {dir_size_gb(cadd_dir):.1f} GB".ljust(w) + "\u2551")
print("\u2560" + "\u2550" * w + "\u2563")

# Overall status
all_ready = ref_ok and vep_src_ok and vep_cache_ok and annovar_ok and db_count == len(protocols)
if all_ready:
    print("\u2551" + "  \u2705 Ready to run 01_vep_annovar_annotation".ljust(w) + "\u2551")
else:
    missing = []
    if not ref_ok: missing.append("reference FASTA")
    if not vep_src_ok: missing.append("VEP source")
    if not vep_cache_ok: missing.append("VEP cache")
    if not annovar_ok: missing.append("ANNOVAR scripts")
    if db_count < len(protocols): missing.append(f"{len(protocols) - db_count} ANNOVAR database(s)")
    print("\u2551" + f"  \u26a0\ufe0f  Missing: {', '.join(missing)}".ljust(w) + "\u2551")
print("\u255a" + "\u2550" * w + "\u255d")

## Troubleshooting

### Common Issues

| Issue | Solution |
| --- | --- |
| **"Permission denied"** | Ensure cluster has internet access and UC Volume write permissions |
| **"VEP cache not found"** | Cache tarball extracts to `homo_sapiens/115_GRCh38/` \~ verify directory structure matches |
| **"ANNOVAR download failed"** | Registration URL may have expired. Get a new one at https://annovar.openbioinformatics.org/ |
| **"samtools not found"** | Install with `sudo apt-get install -y samtools` |
| **"Perl module not found"** | Install missing modules: `sudo cpanm DBI Archive::Zip JSON` |
| **Database download times out** | Re-run the download cell \~ it will skip already-installed databases |
| **"disk space"** | Ensure UC Volume has at least 60 GB free. CADD adds \~30 GB extra. |

### Updating Databases

| Component | How to Update |
| --- | --- |
| **ClinVar** | Change protocol to `clinvar_YYYYMMDD` (e.g. `clinvar_20250101`) and re-run download cell |
| **VEP** | Change git branch to `release/116`, download new cache tarball, update `cache_version` |
| **gnomAD** | New versions: `gnomad41_genome`, `gnomad42_genome`, etc. |
| **CADD** | Download from https://cadd.gs.washington.edu/download for newer versions |
| **Reference genome** | Rarely changes \~ only needed for a new assembly version |

### Disk Cleanup

After successful setup, safe to remove temporary files:
```python
# Clean up /tmp downloads (already done automatically, but just in case)
import os
for f in os.listdir('/tmp'):
    if f.endswith('.tar.gz') and ('vep' in f or 'annovar' in f or 'CADD' in f):
        os.remove(f'/tmp/{f}')
        print(f'Removed /tmp/{f}')
```

### Re-running This Notebook

All download cells check for existing files first and skip if already present. It is safe to re-run the entire notebook \~ only missing components will be downloaded.

## Optional: Patch Streamlit App

The cell below patches `disease_biology.py` to add the **VEP 115 + ANNOVAR** form to the Variant Annotation tab. It makes four targeted insertions:
1. Import the `vep_annovar_annotation` utility module
2. Add `annotation_v2_complete` to the progress map
3. Add `annotation_v2_complete` to the complete statuses set
4. Add the v2 form UI to the Variant Annotation tab

Run this cell **once** after deploying the utility file. Safe to re-run (idempotent).

In [0]:
# ── Patch disease_biology.py with VEP + ANNOVAR v2 integration ───────────────
# Applies targeted insertions — does NOT rewrite the entire file.
# Safe to re-run (checks for existing markers before patching).

import os

SOURCE_FILE = "/Workspace/Users/andrew_forman@eisai.com/genesis-workbench/modules/core/app/views/disease_biology.py"
DEPLOYED_FILE = "/Workspace/Users/andrew_forman@eisai.com/.bundle/genesis_workbench/core/prod/files/app/views/disease_biology.py"

# VEP_ANNOVAR_ANNOTATION_JOB_ID = "633979331375413"  # Created job

def patch_file(filepath):
    """Apply targeted patches to disease_biology.py."""
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()

    patched = False

    # ── Patch 1: Add import for v2 utility ──
    v2_import = "from utils.vep_annovar_annotation import (\n    start_vep_annovar_annotation,\n    search_vep_annovar_runs_by_run_name,\n    search_vep_annovar_runs_by_experiment_name,\n    pull_vep_annovar_results,\n)"
    if "vep_annovar_annotation" not in content:
        # Insert after the last existing import
        anchor = "from genesis_workbench.models import set_mlflow_experiment"
        if anchor in content:
            content = content.replace(anchor, anchor + "\n\n" + v2_import)
            patched = True
            print("  \u2705 Patch 1: Added v2 import")
    else:
        print("  \u23ed\ufe0f  Patch 1: v2 import already present")

    # ── Patch 2: Add annotation_v2_complete to _PROGRESS_MAP ──
    if '"annotation_v2_complete"' not in content:
        # Insert after annotation_complete line
        anchor = '    "annotation_complete":  "\U0001f7e9\U0001f7e9\U0001f7e9",'
        replacement = anchor + '\n    # Variant Annotation v2 (VEP + ANNOVAR)\n    "annotation_v2_complete": "\U0001f7e9\U0001f7e9\U0001f7e9",'
        if anchor in content:
            content = content.replace(anchor, replacement)
            patched = True
            print("  \u2705 Patch 2: Added annotation_v2_complete to _PROGRESS_MAP")
        else:
            print("  \u26a0\ufe0f  Patch 2: Could not find _PROGRESS_MAP anchor")
    else:
        print("  \u23ed\ufe0f  Patch 2: annotation_v2_complete already in _PROGRESS_MAP")

    # ── Patch 3: Add annotation_v2_complete to _COMPLETE_STATUSES ──
    if '"annotation_v2_complete"' not in content or '"annotation_v2_complete"' in content.split('_COMPLETE_STATUSES')[0]:
        old_statuses = '_COMPLETE_STATUSES = {"alignment_complete", "gwas_complete", "ingestion_complete", "annotation_complete"}'
        new_statuses = '_COMPLETE_STATUSES = {"alignment_complete", "gwas_complete", "ingestion_complete", "annotation_complete", "annotation_v2_complete"}'
        if old_statuses in content:
            content = content.replace(old_statuses, new_statuses)
            patched = True
            print("  \u2705 Patch 3: Added annotation_v2_complete to _COMPLETE_STATUSES")
        elif new_statuses in content:
            print("  \u23ed\ufe0f  Patch 3: annotation_v2_complete already in _COMPLETE_STATUSES")
        else:
            print("  \u26a0\ufe0f  Patch 3: Could not find _COMPLETE_STATUSES anchor")

    # ── Patch 4: Add v2 form to variant_annotation_tab ──
    v2_marker = "# ── VEP + ANNOVAR v2 form"
    if v2_marker not in content:
        v2_ui_block = '''

    st.divider()
    # ── VEP + ANNOVAR v2 form ───────────────────────────────────────────────────────
    st.subheader("VEP 115 + ANNOVAR Annotation (v2)")
    st.caption("Full-spectrum variant annotation: functional consequences (VEP), population frequencies, clinical significance, deleteriousness scores (ANNOVAR).")

    with st.form("vep_annovar_form"):
        v2_col1, v2_col2 = st.columns(2)
        with v2_col1:
            v2_experiment = st.text_input("MLflow Experiment Name", value="variant_annotation_v2", key="v2_exp")
            v2_run_name = st.text_input("MLflow Run Name", value="", key="v2_run")
            v2_vcf_path = st.text_input("VCF File Path", placeholder="/Volumes/...", key="v2_vcf")
            v2_genome = st.selectbox("Genome Build", ["GRCh38", "GRCh37"], index=0, key="v2_genome")
        with v2_col2:
            v2_tools = st.selectbox("Annotation Tools", ["both", "vep_only", "annovar_only"], index=0, key="v2_tools",
                                    help="Run VEP only, ANNOVAR only, or both")
            v2_protocols = st.text_input("ANNOVAR Protocols",
                value="refGene,gnomad40_genome,clinvar_20240917,dbnsfp42a,cosmic70", key="v2_protocols")
            v2_operations = st.text_input("ANNOVAR Operations", value="g,f,f,f,f", key="v2_ops")
            v2_vep_flags = st.text_input("VEP Extra Flags (optional)", value="", key="v2_flags")

        if st.form_submit_button(":material/play_arrow: Start VEP + ANNOVAR Annotation", use_container_width=True):
            if v2_run_name and v2_vcf_path:
                with st.spinner("Starting VEP + ANNOVAR annotation..."):
                    try:
                        job_run_id = start_vep_annovar_annotation(
                            user_info=user_info,
                            vcf_path=v2_vcf_path,
                            genome_build=v2_genome,
                            annotation_tools=v2_tools,
                            annovar_protocols=v2_protocols,
                            annovar_operations=v2_operations,
                            vep_extra_flags=v2_vep_flags,
                            mlflow_experiment_name=v2_experiment,
                            mlflow_run_name=v2_run_name,
                        )
                        st.success(f"VEP + ANNOVAR annotation started \u2014 Run ID: `{job_run_id}`")
                    except Exception as e:
                        st.error(f"Failed: {e}")

    st.divider()
    st.subheader("Search VEP + ANNOVAR Runs")
    v2s_col1, v2s_col2 = st.columns(2)
    with v2s_col1:
        v2_search_by = st.radio("Search by", ["Run Name", "Experiment Name"], horizontal=True, key="v2_search_by")
    with v2s_col2:
        v2_search_query = st.text_input("Search", placeholder="Type to search...", key="v2_search_query")

    if v2_search_query:
        with st.spinner("Searching..."):
            if v2_search_by == "Run Name":
                v2_results = search_vep_annovar_runs_by_run_name(user_info.user_email, v2_search_query)
            else:
                v2_results = search_vep_annovar_runs_by_experiment_name(user_info.user_email, v2_search_query)

        if len(v2_results) > 0:
            v2_results = add_progress_column(v2_results, total_steps=3)
            st.markdown(render_runs_html_table(v2_results, hidden_columns=["run_id", "progress"]), unsafe_allow_html=True)

            # Show results for completed runs
            completed = v2_results[v2_results["status"] == "annotation_v2_complete"]
            if len(completed) > 0:
                v2_run_options = {f"{row[\"run_name\"]} ({row[\"start_time\"]})": row["run_id"] for _, row in completed.iterrows()}
                v2_selected = st.selectbox("View results for", list(v2_run_options.keys()), key="v2_result_select")
                if v2_selected:
                    v2_run_id = v2_run_options[v2_selected]
                    v2_run_name_display = v2_selected.split(" (")[0]
                    with st.spinner("Loading annotation results..."):
                        ann_df = pull_vep_annovar_results(v2_run_id, v2_run_name_display)
                    if len(ann_df) > 0:
                        # Impact summary
                        if "combined_impact" in ann_df.columns:
                            impact_counts = ann_df["combined_impact"].value_counts()
                            imp_cols = st.columns(min(4, len(impact_counts)))
                            for i, (impact, count) in enumerate(impact_counts.items()):
                                imp_cols[i % len(imp_cols)].metric(impact, f"{count:,}")
                        st.dataframe(ann_df, use_container_width=True, height=400)
                    else:
                        st.caption("No annotation results found for this run.")
        else:
            st.caption("No VEP + ANNOVAR runs found.")
'''
        # Find the end of the variant_annotation_tab content to insert before the end
        # The existing tab ends with the v1 form submit button block
        v1_end_marker = '                    except Exception as e:\n                        st.error(f"Failed: {e}")\n'
        # Count occurrences to find the LAST one (which is in variant_annotation_tab)
        last_idx = content.rfind(v1_end_marker)
        if last_idx > -1:
            # Find the end of this block (closing of the form)
            insert_pos = last_idx + len(v1_end_marker)
            content = content[:insert_pos] + v2_ui_block + content[insert_pos:]
            patched = True
            print("  \u2705 Patch 4: Added VEP + ANNOVAR v2 form to variant_annotation_tab")
        else:
            # Fallback: append to end of file
            content += v2_ui_block
            patched = True
            print("  \u2705 Patch 4: Appended VEP + ANNOVAR v2 form (fallback: end of file)")
    else:
        print("  \u23ed\ufe0f  Patch 4: VEP + ANNOVAR v2 form already present")

    if patched:
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(content)
        print(f"\n\u2705 Patched: {filepath}")
    else:
        print(f"\n\u23ed\ufe0f  No changes needed: {filepath}")

    return patched


# ── Apply patches ──
print("Patching disease_biology.py for VEP + ANNOVAR v2 ...\n")
print("Source file:")
patch_file(SOURCE_FILE)

if os.path.isfile(DEPLOYED_FILE):
    print(f"\nDeployed file:")
    patch_file(DEPLOYED_FILE)
else:
    print(f"\n\u26a0\ufe0f  Deployed file not found: {DEPLOYED_FILE}")
    print("  Run deploy.sh or copy manually after patching source.")

print(f"\n\u2500\u2500 Job IDs (set these in application.env) \u2500\u2500")
print(f"VEP_ANNOVAR_ANNOTATION_JOB_ID=633979331375413")
print(f"VARIANT_ANNOTATION_V2_SETUP_JOB_ID=930917968034893")